# Hackathon Baseline Model 

To predict **binarised protein expression**.

- Targets binarised using per-protein median threshold (expression > median → 1, else → 0) to test it works with binarised outputs
- Multi-label binary classification
- One score per class per protein
- Loss: `SparseCategoricalCrossentropy(from_logits=True)`
- Evaluation uses accuracy
- Both model architectures still relatively intact from kaggle competition 2nd place: [2nd Place GRU CITE](https://www.kaggle.com/code/senkin13/2nd-place-gru-cite)
- Data from [GSE194315](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE194315)

# Imports

In [1]:
import numpy as np
import pandas as pd
import gc
import muon as mu
from scipy.sparse import issparse, diags, csr_matrix
from sklearn.decomposition import TruncatedSVD
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

# Change this to your local path where the data files are stored
DATA_PATH = '/Users/teosecilmis/Documents/CITESEQ/Teo_datasets/Hackathon files/'

# Number of SVD components for RNA input features (to avoid memory issues)
N_SVD_COMPONENTS = 1000

# Subsample to avoid memory issues during training 
SUBSAMPLE_N = 30000

/opt/anaconda3/envs/teo_citeprep_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Raw Data

In [2]:
# Load MuData (RNA + ADT combined, isotype controls already removed)
mdata = mu.read(DATA_PATH + 'GSE194315_raw_mu.h5mu')
print(f'MuData: {mdata.shape}')
print(f'Modalities: {list(mdata.mod.keys())}')

# Extract RNA and ADT modalities
rna_adata = mdata.mod['rna']
adt_adata = mdata.mod['protein']
print(f'RNA: {rna_adata.shape} (cells x genes), sparse={issparse(rna_adata.X)}')
print(f'ADT: {adt_adata.shape} (cells x proteins)')

del mdata; gc.collect()

if SUBSAMPLE_N is not None and SUBSAMPLE_N < rna_adata.n_obs:
    np.random.seed(42)
    idx = np.sort(np.random.choice(rna_adata.n_obs, SUBSAMPLE_N, replace=False))
    rna_adata = rna_adata[idx].copy()
    adt_adata = adt_adata[idx].copy()
    print(f'\nSubsampled to {SUBSAMPLE_N} cells')

# Extract ADT targets as dense array 
train_cite_y = adt_adata.X
if issparse(train_cite_y):
    train_cite_y = train_cite_y.toarray()
train_cite_y = train_cite_y.astype(np.float32)
protein_names = adt_adata.var_names.values.astype(str)
del adt_adata; gc.collect()

# Remove cells with constant protein values (dead cells)
row_stds = train_cite_y.std(axis=1)
valid_mask = row_stds > 0
n_removed = (~valid_mask).sum()
train_cite_y = train_cite_y[valid_mask]
rna_adata = rna_adata[valid_mask].copy()
print(f'Removed {n_removed} constant cells, {valid_mask.sum()} remaining')

N_TARGETS = train_cite_y.shape[1]

# Binarise targets
thresholds = np.median(train_cite_y, axis=0)  # shape (273,)
train_cite_y = (train_cite_y > thresholds).astype(np.float32)
print(f'\nBinarised targets with per-protein median threshold')
print(f'Class balance (fraction "on"): {train_cite_y.mean():.3f}')
print(f'Per-protein fraction "on": {train_cite_y.mean(axis=0).min():.2f} to {train_cite_y.mean(axis=0).max():.2f}')

train_df = pd.DataFrame({'cell_id': rna_adata.obs_names.values.astype(str)})

print(f'\nTrain RNA:  {rna_adata.shape}  (cells x genes)')
print(f'Train ADT:  {train_cite_y.shape}  (cells x proteins)')
print(f'Protein names:  {protein_names[:5]} ...')

/opt/anaconda3/envs/teo_citeprep_env/lib/python3.11/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/opt/anaconda3/envs/teo_citeprep_env/lib/python3.11/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData: (180794, 33967)
Modalities: ['rna', 'protein']
RNA: (180794, 33694) (cells x genes), sparse=True
ADT: (180794, 273) (cells x proteins)

Subsampled to 30000 cells
Removed 7792 constant cells, 22208 remaining

Binarised targets with per-protein median threshold
Class balance (fraction "on"): 0.352
Per-protein fraction "on": 0.00 to 0.50

Train RNA:  (22208, 33694)  (cells x genes)
Train ADT:  (22208, 273)  (cells x proteins)
Protein names:  ['C5L2' 'Cadherin11' 'CCR10-1' 'CD101|BB27' 'CD102|ICAM-2'] ...


# RNA Input Preprocessing

**Pipeline:** raw counts → library size normalization → log1p → TruncatedSVD

Normalization is done on the **sparse matrix** to avoid densifying the full 180k × 33k RNA
matrix (~24 GB dense vs ~2 GB sparse). The math is identical:
- Library size norm: divide each cell by its total counts, scale to 10,000
- Log1p
- TruncatedSVD 

Remove during hackathon as we will have already normalized data

In [3]:
def normalize_rna_sparse(X):
    """Library size normalization + log1p for sparse matrices.
    """
    if not issparse(X):
        X = csr_matrix(X)
    X = X.astype(np.float32).copy()

    # Library size normalization: divide each cell by total counts, scale to 10,000
    lib_sizes = np.array(X.sum(axis=1)).flatten()
    lib_sizes[lib_sizes == 0] = 1  # avoid division by zero for empty cells
    X = diags(1e4 / lib_sizes) @ X

    # Log1p
    X = X.tocsr()
    X.data = np.log1p(X.data)

    return X

print('Normalizing RNA (library size norm + log1p, sparse)...')
rna_sparse = normalize_rna_sparse(rna_adata.X)
del rna_adata; gc.collect()

print(f'Fitting TruncatedSVD with {N_SVD_COMPONENTS} components on {rna_sparse.shape[1]} genes...')
svd = TruncatedSVD(n_components=N_SVD_COMPONENTS, random_state=42)
train_cite_X = svd.fit_transform(rna_sparse).astype(np.float32)
print(f'SVD explained variance ratio (cumulative): {svd.explained_variance_ratio_.sum():.4f}')

del rna_sparse, svd; gc.collect()

print(f'Train features: {train_cite_X.shape}')

Normalizing RNA (library size norm + log1p, sparse)...
Fitting TruncatedSVD with 1000 components on 33694 genes...
SVD explained variance ratio (cumulative): 0.4506
Train features: (22208, 1000)


In [4]:
train_df.shape, train_cite_X.shape, train_cite_y.shape

((22208, 1), (22208, 1000), (22208, 273))

# Utils (helper functions)

In [ ]:
N_CLASSES = 2

def classification_accuracy(y_true, y_pred_proba, threshold=0.5):
    """Compute accuracy for multi-label binary classification."""
    y_pred_binary = (y_pred_proba >= threshold).astype(np.float32)
    return accuracy_score(y_true.flatten(), y_pred_binary.flatten())


class DataGenerator(tf.keras.utils.Sequence):
    'Generates data for Keras'
    def __init__(self, train_X, train_y, list_IDs, shuffle, batch_size, labels):
        self.train_X = train_X
        self.train_y = train_y
        self.list_IDs = list_IDs
        self.shuffle = shuffle
        self.batch_size = batch_size
        self.labels = labels
        self.on_epoch_end()

    def __len__(self):
        ct = len(self.list_IDs) // self.batch_size
        return ct

    def __getitem__(self, idx):
        indexes = self.indexes[idx*self.batch_size:(idx+1)*self.batch_size]
        list_IDs_temp = [self.list_IDs[k] for k in indexes]
        X, y = self.__data_generation(list_IDs_temp)
        if self.labels: return X, y
        else: return X

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.list_IDs))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, list_IDs_temp):
        X = self.train_X[list_IDs_temp]
        y = self.train_y[list_IDs_temp]
        return X, y


def nn_kfold(train_df, train_cite_X, train_cite_y, network, folds, model_name):
    oof_preds = np.zeros((train_df.shape[0], N_TARGETS))
    cv_metrics = []
    for n_fold, (train_idx, valid_idx) in enumerate(folds.split(train_df,)):
        print(f'\n=== Fold {n_fold} ===')
        train_x = train_cite_X[train_idx]
        valid_x = train_cite_X[valid_idx]
        train_y = train_cite_y[train_idx]
        valid_y = train_cite_y[valid_idx]

        train_x_index = train_df.iloc[train_idx].reset_index(drop=True).index
        valid_x_index = train_df.iloc[valid_idx].reset_index(drop=True).index

        model = network(train_cite_X.shape[1])
        filepath = model_name + '_' + str(n_fold) + '.weights.h5'
        es = tf.keras.callbacks.EarlyStopping(patience=10, mode='min', verbose=1)
        checkpoint = tf.keras.callbacks.ModelCheckpoint(
            monitor='val_loss', filepath=filepath,
            save_best_only=True, save_weights_only=True, mode='min'
        )
        reduce_lr_loss = tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=LR_FACTOR, patience=6, verbose=1
        )

        train_dataset = DataGenerator(
            train_x, train_y,
            list_IDs=train_x_index,
            shuffle=True,
            batch_size=BATCH_SIZE,
            labels=True,
        )

        valid_dataset = DataGenerator(
            valid_x, valid_y,
            list_IDs=valid_x_index,
            shuffle=False,
            batch_size=BATCH_SIZE,
            labels=True,
        )

        hist = model.fit(
            train_dataset,
            validation_data=valid_dataset,
            epochs=EPOCHS,
            callbacks=[checkpoint, es, reduce_lr_loss],
            verbose=1
        )

        model.load_weights(filepath)

        # Model outputs logits of shape (batch, N_TARGETS, 2)
        # Apply softmax and extract probability of class 1 (expressed)
        raw_preds = model.predict(valid_x, batch_size=BATCH_SIZE, verbose=1)
        probs = tf.nn.softmax(raw_preds, axis=-1).numpy()
        oof_preds[valid_idx] = probs[:, :, 1]

        acc = classification_accuracy(valid_y, oof_preds[valid_idx])
        cv_metrics.append(acc)
        print(f'Fold {n_fold} — Acc: {acc:.4f}')

        del model
        gc.collect()
        tf.keras.backend.clear_session()

    acc = classification_accuracy(train_cite_y, oof_preds)
    print(f'\nOverall — Acc: {acc:.4f}')
    return oof_preds

# Model 1 — GRU + 2-class output (adapted from Cosine Similarity model from intitial Kaggle comp)

In [6]:
def cite_clf_gru_model(len_num):

    input_num = tf.keras.Input(shape=(len_num,))
    x = input_num
    x0 = tf.keras.layers.Reshape((1, x.shape[1]))(x)
    x0 = tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(1800, activation='elu', kernel_initializer='identity', return_sequences=False)
    )(x0)
    x1 = tf.keras.layers.GaussianDropout(0.2)(x0)
    x2 = tf.keras.layers.Dense(1800, activation='elu', kernel_initializer='identity')(x1)
    x3 = tf.keras.layers.GaussianDropout(0.2)(x2)
    x4 = tf.keras.layers.Dense(1800, activation='elu', kernel_initializer='identity')(x3)
    x5 = tf.keras.layers.GaussianDropout(0.2)(x4)
    x = tf.keras.layers.Concatenate()([x1, x3, x5])
    # Linear projection to N_TARGETS * N_CLASSES, then reshape to (N_TARGETS, 2)
    x = tf.keras.layers.Dense(N_TARGETS * N_CLASSES)(x)
    output = tf.keras.layers.Reshape((N_TARGETS, N_CLASSES))(x)

    model = tf.keras.models.Model(input_num, output)
    lr = 0.001
    adam = tf.keras.optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.999)
    model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), optimizer=adam)
    model.summary()
    return model


BATCH_SIZE = 620
EPOCHS = 100
LR_FACTOR = 0.05
SEED = 666
N_FOLD = 5
folds = KFold(n_splits=N_FOLD, shuffle=True, random_state=SEED)

oof_preds_m1 = nn_kfold(
    train_df, train_cite_X, train_cite_y,
    cite_clf_gru_model, folds, 'raw_cite_clf_gru_model'
)


=== Fold 0 ===


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1000)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1000)   │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 3600)      │ 30,261,600 │ reshape[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_dropout    │ (None, 3600)      │          0 │ bidirectional[0]… │
│ (GaussianDropout)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1800)      │  6,481,800 │ gaussian_dropout… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_dropout_1  │ (None, 1800)      │          0 │ dense[0][0]       │
│ (GaussianDropout)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1800)      │  3,241,800 │ gaussian_dropout… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_dropout_2  │ (None, 1800)      │          0 │ dense_1[0][0]     │
│ (GaussianDropout)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 7200)      │          0 │ gaussian_dropout… │
│ (Concatenate)       │                   │            │ gaussian_dropout… │
│                     │                   │            │ gaussian_dropout… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 546)       │  3,931,746 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 273, 2)    │          0 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 43,916,946 (167.53 MB)

 Trainable params: 43,916,946 (167.53 MB)

...

 Trainable params: 43,916,946 (167.53 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100


/opt/anaconda3/envs/teo_citeprep_env/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


28/28 ━━━━━━━━━━━━━━━━━━━━ 12s 347ms/step - loss: 0.5842 - val_loss: 0.5193 - learning_rate: 0.0010
Epoch 2/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 9s 316ms/step - loss: 0.4981 - val_loss: 0.4977 - learning_rate: 0.0010
Epoch 3/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 9s 314ms/step - loss: 0.4747 - val_loss: 0.4955 - learning_rate: 0.0010
Epoch 4/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 9s 309ms/step - loss: 0.4570 - val_loss: 0.4957 - learning_rate: 0.0010
Epoch 5/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 9s 306ms/step - loss: 0.4435 - val_loss: 0.5011 - learning_rate: 0.0010
Epoch 6/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 9s 317ms/step - loss: 0.4279 - val_loss: 0.5056 - learning_rate: 0.0010
Epoch 7/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 9s 321ms/step - loss: 0.4100 - val_loss: 0.5164 - learning_rate: 0.0010
Epoch 8/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 9s 316ms/step - loss: 0.4070 - val_loss: 0.5277 - learning_rate: 0.0010
Epoch 9/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step - loss: 0.3949
Epoch 9: ReduceLROnPlateau reducing learning rate to 5.

# Model 2 — Dense/GRU + 2-class output (adapted from MSE model)

In [7]:
def cite_clf_dense_model(len_num):

    input_num = tf.keras.Input(shape=(len_num,))
    x = input_num
    x = tf.keras.layers.Dense(1500, activation='swish')(x)
    x = tf.keras.layers.GaussianDropout(0.1)(x)
    x = tf.keras.layers.Dense(1500, activation='swish')(x)
    x = tf.keras.layers.GaussianDropout(0.1)(x)
    x = tf.keras.layers.Dense(1500, activation='swish')(x)
    x = tf.keras.layers.GaussianDropout(0.1)(x)
    x = tf.keras.layers.Reshape((1, x.shape[1]))(x)
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(700, activation='swish', return_sequences=False)
    )(x)
    x = tf.keras.layers.GaussianDropout(0.1)(x)
    x = tf.keras.layers.Dense(N_TARGETS * N_CLASSES)(x)
    output = tf.keras.layers.Reshape((N_TARGETS, N_CLASSES))(x)

    model = tf.keras.models.Model(input_num, output)
    lr = 0.0005
    weight_decay = 0.0001
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), optimizer=opt)
    model.summary()
    return model


BATCH_SIZE = 600
EPOCHS = 100
LR_FACTOR = 0.1
SEED = 666
folds = KFold(n_splits=5, shuffle=True, random_state=SEED)

oof_preds_m2 = nn_kfold(
    train_df, train_cite_X, train_cite_y,
    cite_clf_dense_model, folds, 'raw_cite_clf_dense_model'
)


=== Fold 0 ===


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1500)           │     1,501,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gaussian_dropout                │ (None, 1500)           │             0 │
│ (GaussianDropout)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1500)           │     2,251,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gaussian_dropout_1              │ (None, 1500)           │             0 │
│ (GaussianDropout)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1500)           │     2,251,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gaussian_dropout_2              │ (None, 1500)           │             0 │
│ (GaussianDropout)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 1, 1500)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 1400)           │     9,248,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gaussian_dropout_3              │ (None, 1400)           │             0 │
│ (GaussianDropout)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 546)            │       764,946 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 273, 2)         │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,017,846 (61.10 MB)

 Trainable params: 16,017,846 (61.10 MB)

...

 Trainable params: 16,017,846 (61.10 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100


/opt/anaconda3/envs/teo_citeprep_env/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


29/29 ━━━━━━━━━━━━━━━━━━━━ 7s 149ms/step - loss: 0.5822 - val_loss: 0.5520 - learning_rate: 5.0000e-04
Epoch 2/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 136ms/step - loss: 0.5448 - val_loss: 0.5284 - learning_rate: 5.0000e-04
Epoch 3/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 138ms/step - loss: 0.5301 - val_loss: 0.5124 - learning_rate: 5.0000e-04
Epoch 4/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 139ms/step - loss: 0.5130 - val_loss: 0.5081 - learning_rate: 5.0000e-04
Epoch 5/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 140ms/step - loss: 0.5009 - val_loss: 0.4968 - learning_rate: 5.0000e-04
Epoch 6/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 136ms/step - loss: 0.4967 - val_loss: 0.5027 - learning_rate: 5.0000e-04
Epoch 7/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 141ms/step - loss: 0.4963 - val_loss: 0.4904 - learning_rate: 5.0000e-04
Epoch 8/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 143ms/step - loss: 0.4850 - val_loss: 0.4845 - learning_rate: 5.0000e-04
Epoch 9/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 136ms/step - loss: 0.4789 - val_loss: 0.4846 - lea

# Blending & Final Evaluation (just like was done in comp)

In [8]:
# Average predicted probabilities from both models
oof_preds = oof_preds_m1 * 0.55 + oof_preds_m2 * 0.45
acc = classification_accuracy(train_cite_y, oof_preds)
print(f'Blend — Acc: {acc:.4f}')

Blend — Acc: 0.7753


In [9]:
del train_df, train_cite_X, train_cite_y
del oof_preds_m1, oof_preds_m2
gc.collect()

3847